In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
%run "../shared/00.common-variables"

In [0]:
def generate_pipeline_run_id(pipeline_name, start_time):
    return (f"{pipeline_name}_" + start_time.strftime("%Y%m%d_%H%M%S"))

In [0]:
def read_bronze_source(schema_name, source_file_path):
    return (spark.read.format('csv')\
                .option("header",True)\
                .option("mode","PERMISSIVE")\
                .option("columnNameOfCorruptRecord", "_corrupt_record")\
                .schema(schema_name)\
                .load(source_file_path))

In [0]:
def add_ingestion_metadata(df, pipeline_run_id):
    return (df.withColumn("ingestion_timestamp",F.current_timestamp())\
                     .withColumn("source_file_name", F.col("_metadata.file_path"))\
                     .withColumn("source_system", F.lit("olist"))\
                     .withColumn("pipeline_run_id", F.lit(pipeline_run_id)))
    
# _metadata = Part of databricks
# While you can often reference columns using string names or dot notation (e.g., df.age), using col() is the Spark native way to create expressions that are not yet bound to a specific DataFrame, making the code more flexible

In [0]:
def calculate_dq_metrics(df,business_key):

    source_row_count = df.count() ## rows read during current run of the pipeline

    corrupt_row_count = df.filter(
        F.col("_corrupt_record").isNotNull()
    ).count()

    if business_key != None:
        null_business_key_count = df.filter(
            F.col(f"{business_key}").isNull()
        ).count()

        # No. of duplicated order IDs
        duplicated_business_key_count = (
            df
            .groupBy(f"{business_key}")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
    else:
        null_business_key_count = None
        duplicated_business_key_count = None

    target_row_count = source_row_count

    return (source_row_count, target_row_count, null_business_key_count, duplicated_business_key_count, corrupt_row_count)

In [0]:
def write_logs(logs_df, full_logs_table_name):
    logs_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(full_logs_table_name)

In [0]:
import hashlib

def generate_file_checksum(file_path, algorithm='sha256'):
    # Select the hash algorithm (e.g., 'sha256', 'md5')
    hash_func = hashlib.new(algorithm)
    
    # Read the file in binary chunks to avoid out-of-memory errors
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hash_func.update(chunk)
            
    return hash_func.hexdigest()


In [0]:
file_path = "/Volumes/olist/landing/files/olist_orders_dataset.csv"
print(generate_file_checksum(file_path))